In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"

bronzeTable = f"{catalog}.bronze_schema_{env}.rawInvoice"
silverTable = f"{catalog}.silver_schema_{env}.cleanedInvoice"
print(f"Source  : {bronzeTable}")
print(f"Target  : {silverTable}")

srcFileLoc=f"/Volumes/{catalog}/silver_schema_{env}/vol_saleslake_src_files_{env}/invoice/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTable}
SELECT DISTINCT
    TRIM(invoice_id)                                              AS invoice_id,
    TRIM(invoice_number)                                          AS invoice_number,
    CAST(TRIM(customer_id) AS BIGINT)                             AS customer_id,
    TO_DATE(TRIM(invoice_date),  'yyyy-MM-dd')                    AS invoice_date,
    TO_DATE(TRIM(due_date),      'yyyy-MM-dd')                    AS due_date,
    CAST(TRIM(subtotal_amount) AS DECIMAL(18,2))                  AS subtotal_amount,
    UPPER(NULLIF(TRIM(discount_code), ''))                        AS discount_code,
    CAST(TRIM(discount_amount) AS DECIMAL(18,2))                  AS discount_amount,
    CAST(TRIM(tax_amount)      AS DECIMAL(18,2))                  AS tax_amount,
    CAST(TRIM(total_amount)    AS DECIMAL(18,2))                  AS total_amount,
    UPPER(TRIM(payment_status))                                   AS payment_status,
    UPPER(NULLIF(TRIM(payment_method), ''))                       AS payment_method,
    CASE
        WHEN NULLIF(TRIM(payment_date), '') IS NULL THEN NULL
        ELSE TO_DATE(TRIM(payment_date), 'yyyy-MM-dd')
    END                                                           AS payment_date,
    UPPER(TRIM(currency))                                         AS currency,
    UPPER(TRIM(region))                                           AS region,
    UPPER(TRIM(store_id))                                         AS store_id,
    UPPER(TRIM(channel))                                          AS channel,
    LOWER(TRIM(created_by))                                       AS created_by,
    CURRENT_TIMESTAMP()                                           AS ingest_ts
FROM {bronzeTable}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP('1990-01-01','yyyy-MM-dd'))
    FROM {silverTable}
)
ORDER BY TRIM(invoice_id)
""")

In [0]:
# Invoice Count
spark.sql(f"""
SELECT COUNT(*) AS total_invoices
FROM {silverTable}
""").show()

In [0]:
# Duplicate Invoice Validation
spark.sql(f"""
SELECT
    invoice_id,
    COUNT(*) AS duplicate_count
FROM {silverTable}
GROUP BY invoice_id
HAVING COUNT(*) > 1
""").show()

In [0]:
# Negative Amount Validation
spark.sql(f"""
SELECT
    invoice_id,
    subtotal_amount,
    discount_amount,
    tax_amount,
    total_amount
FROM {silverTable}
WHERE subtotal_amount < 0
OR discount_amount < 0
OR tax_amount < 0
OR total_amount < 0
""").show()

In [0]:
# Invoice Amount Calculation Validation
# Business rule:
# subtotal_amount - discount_amount + tax_amount = total_amount
# Validation query:

spark.sql(f"""
SELECT
    invoice_id,
    subtotal_amount,
    discount_amount,
    tax_amount,
    total_amount
FROM {silverTable}
WHERE ROUND(
        subtotal_amount - discount_amount + tax_amount,
        2
      )
      <> ROUND(total_amount, 2)
""").show()